# NB11 — Historical human-label classifier pilot

Notebook này dùng lại **human ground truth đã có** thay vì tiếp tục đoán threshold hoặc review thêm ngay:

- confirmed duplicate queue: 400 rows cũ, dedup theo policy calibration còn khoảng 322 visual positives;
- hard-negative review: 300 rows đã human-label;
- theo definition hiện tại của project, `SAME_PRODUCT_DIFFERENT_IMAGE` được map thành `DUPLICATE` (cùng visual item, chỉ khác góc/ánh sáng rất nhẹ).

Sau đó notebook extract cùng một bộ cheap preview-features cho tất cả pair, split **group-disjoint theo E3 image path**, train Logistic Regression / RBF-SVM / Random Forest, chọn model bằng validation, rồi test trên held-out test.

Đây vẫn là **pilot**, chưa phải canonical contamination detector. Mục tiêu là xem classical ML có học được boundary đáng tin hay không trước khi chạy full E3.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive, auth
    drive.mount('/content/drive', force_remount=False)
    auth.authenticate_user()
except ImportError:
    pass

REPO_URL='https://github.com/ThinhTran2208/opisoverated.git'
BRANCH='feat/evaluation3-active-learning-nb11'
REPO_ROOT=Path('/content/opisoverated-e3-nb11-historical')
def run_git(*args,cwd=None):
    return subprocess.run(['git','-c','http.version=HTTP/1.1',*args],cwd=cwd,check=True,text=True)
if not (REPO_ROOT/'.git').is_dir():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    run_git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_ROOT))
else:
    run_git('fetch','origin',BRANCH,cwd=REPO_ROOT); run_git('switch',BRANCH,cwd=REPO_ROOT); run_git('pull','--ff-only','origin',BRANCH,cwd=REPO_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO_ROOT/'requirements-evaluation.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','gspread>=6,<7'],check=True)
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))

from src.evaluation.evaluation3_historical_classifier import (
    prepare_historical_metadata, build_historical_feature_dataset,
    split_historical_dataset, split_summary,
)
from src.evaluation.evaluation3_preview_active_learning import fit_models, xframe, attach_labels
from src.evaluation.evaluation3_active_learning import (
    binary_metrics, choose_triage_thresholds, apply_triage,
)
from IPython.display import display
print('BRANCH:',BRANCH)


## 1. Load historical human labels

Cần folder `phash_ssim_threshold` xuất hiện trong `MyDrive` (nếu đây là shared folder thì Add shortcut to My Drive). Notebook đọc label hard-negative trực tiếp từ Google Sheet đã review.


In [ ]:
DRIVE=Path('/content/drive/MyDrive')
ROOT_CANDIDATES=[
    DRIVE/'phash_ssim_threshold',
    DRIVE/'EVALUATION3'/'phash_ssim_threshold',
]
HIST_ROOT=next((p for p in ROOT_CANDIDATES if (p/'confirmed_duplicates_dedup.csv').is_file()),None)
if HIST_ROOT is None:
    raise FileNotFoundError(
        'Không thấy phash_ssim_threshold trong MyDrive. Hãy Add shortcut shared folder vào My Drive, hoặc sửa HIST_ROOT thủ công.'
    )

HARD_REVIEW_SHEET_ID='1b9pj-E_C0BMfyN8s6upKxzcfntR_T77QAfLMwBDBu_g'
import google.auth, gspread
creds,_=google.auth.default()
gc=gspread.authorize(creds)
ws=gc.open_by_key(HARD_REVIEW_SHEET_ID).worksheet('hard_negative_review_BLIND')
hard_labels=pd.DataFrame(ws.get_all_records())

metadata=prepare_historical_metadata(
    historical_root=HIST_ROOT,
    hard_review_labels=hard_labels,
)
print('HIST_ROOT:',HIST_ROOT)
print('Historical rows after dedup:',len(metadata))
print('Labels:',metadata.human_label.value_counts().to_dict())
print('Original hard labels:',hard_labels.human_label.astype(str).str.upper().value_counts().to_dict())
print('Groups (unique E3 image paths):',metadata.group_id.nunique())


## 2. Extract/cache preview features

Lần đầu sẽ đọc khoảng ~600 preview JPG và có thể mất vài phút vì Google Drive I/O. Sau đó cache CSV nên rerun gần như tức thì. Không chạy 28k outfits.


In [ ]:
WORK_DIR=DRIVE/'evaluation3_active_learning_nb11_historical'
WORK_DIR.mkdir(parents=True,exist_ok=True)
FEATURE_CACHE=WORK_DIR/'historical_preview_features.csv'
features=build_historical_feature_dataset(
    metadata, historical_root=HIST_ROOT, cache_csv=FEATURE_CACHE, workers=4
)
print('Feature rows:',len(features))
print('Labels:',features.human_label.value_counts().to_dict())
display(features.head())


## 3. Fixed group-disjoint TRAIN / VALIDATION / TEST

Split khoảng 60/20/20. Cùng `eval3_rel_path` không bao giờ xuất hiện ở hơn một split, kể cả khi E3 image đó có nhiều Polyvore candidates.


In [ ]:
RANDOM_STATE=42
train,val,test=split_historical_dataset(features,random_state=RANDOM_STATE)
summary=split_summary(('train',train),('validation',val),('test',test))
display(summary)
assert set(train.group_id).isdisjoint(set(val.group_id))
assert set(train.group_id).isdisjoint(set(test.group_id))
assert set(val.group_id).isdisjoint(set(test.group_id))
print('Group leakage check: PASS')


## 4. TRAIN — LR / RBF-SVM / Random Forest

Chạy cell này để train cả 3 model trên TRAIN, rank bằng VALIDATION, rồi chọn hai triage thresholds trên VALIDATION.


In [ ]:
models,report=fit_models(train,val,RANDOM_STATE)
display(report)
BEST_NAME=str(report.iloc[0].model)
BEST_MODEL=models[BEST_NAME]
idx=list(BEST_MODEL.classes_).index(1)
val_prob=BEST_MODEL.predict_proba(xframe(val))[:,idx]
THRESHOLDS=choose_triage_thresholds(
    val.target.astype(int).to_numpy(), val_prob,
    target_auto_duplicate_precision=0.98,
    target_auto_non_npv=0.98,
    minimum_auto_examples=10,
)
print('BEST MODEL:',BEST_NAME)
print('TRIAGE THRESHOLDS:',THRESHOLDS)
print('Interpretation: p <= low => AUTO NON; p >= high => AUTO DUP; middle => MANUAL')


## 5. TEST — held-out historical test + optional current hard check

Chỉ chạy sau TRAIN. Đây là **pilot test**, không phải claim cuối cùng của project. Cell cũng thử evaluate trên `round_01_query` hiện tại nếu bạn đã label nó, vì batch này là một current-domain hard set rất hữu ích.


In [ ]:
from sklearn.metrics import confusion_matrix

def evaluate_named(name,frame):
    if len(frame)==0 or frame.target.nunique()<2:
        print(name,': không đủ 2 class để đánh giá đáng tin')
        return
    p=BEST_MODEL.predict_proba(xframe(frame))[:,idx]
    print('\n===',name,'===')
    print('rows / labels:',len(frame),frame.human_label.value_counts().to_dict())
    print('binary metrics:',binary_metrics(frame.target.astype(int).to_numpy(),p))
    pred=(p>=0.5).astype(int)
    print('confusion matrix [[TN,FP],[FN,TP]]:')
    print(confusion_matrix(frame.target.astype(int).to_numpy(),pred,labels=[0,1]))
    tri=apply_triage(p,THRESHOLDS)
    tri_s=pd.Series(tri,index=frame.index)
    print('triage counts:',tri_s.value_counts().to_dict())
    auto_dup=tri_s=='DUPLICATE'; auto_non=tri_s=='NON_DUPLICATE'
    if auto_dup.any(): print('AUTO DUP precision:',float(frame.loc[auto_dup,'target'].mean()),'n=',int(auto_dup.sum()))
    if auto_non.any(): print('AUTO NON NPV:',float((1-frame.loc[auto_non,'target']).mean()),'n=',int(auto_non.sum()))

evaluate_named('HISTORICAL HELD-OUT TEST',test)

# Optional: evaluate on the 30 hard current-domain pairs you already reviewed in round_01.
FAST_DIR=DRIVE/'evaluation3_active_learning_nb11_fast'
fast_features=FAST_DIR/'preview_features_600.csv'
round1=FAST_DIR/'round_01_query.xlsx'
if fast_features.is_file() and round1.is_file():
    current_pool=pd.read_csv(fast_features)
    current=attach_labels(current_pool,[round1])
    current=current[current.target.notna()].copy()
    evaluate_named('CURRENT ROUND_01 HARD CHECK',current)
else:
    print('\nCurrent hard check skipped: không thấy preview_features_600.csv + round_01_query.xlsx')


## Cách đọc kết quả

Nếu historical held-out rất cao **nhưng** current Round-01 hard check tụt mạnh, nghĩa là 400 positive/300 hard-negative cũ chưa đại diện đủ cho near-duplicate distribution hiện tại. Khi đó active learning trên current pool vẫn cần thiết.

Nếu cả hai đều tốt, ta mới có lý do mạnh để tiếp tục classifier và dùng active learning chỉ cho vùng uncertainty thay vì review hàng nghìn ảnh.
